In [3]:
# ── Core libraries ────────────────────────────────────────────────────────────
import os
import re
import string
import warnings
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.color":       "#e8e8e8",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})

PALETTE    = {"ham": "#1D9E75", "spam": "#D85A30"}
STOP_WORDS = set(stopwords.words('english'))

print(" All libraries imported successfully!")


 All libraries imported successfully!


In [4]:
# data loading
data_path =r'/workspaces/Spam-Classifier/data/raw/SMS-Spam'
print(f"Files exists:{os.path.exists(data_path)}")

Files exists:True


In [5]:
# read data
df = pd.read_csv(data_path, sep="\t", header=None, names=["label", "text"])

print(f"(Rows, Columns)  : {df.shape}")
print(f"Columns    : {df.columns.tolist()}")
df.head()

(Rows, Columns)  : (5572, 2)
Columns    : ['label', 'text']


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
#  Data types & memory usage 
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

label    str
text     str
dtype: object

Memory usage: 1000.7 KB


In [7]:
  # class distribution

print("label distribution:")
print(df['label'].value_counts())
print(f"\nSpam ratio: {df['label'].eq('spam').mean()*100:.1f}%")


label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Spam ratio: 13.4%


CLEANING START

In [8]:
# CHECK MISSING VALUES
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nShape after null drop: {df.shape}")
print("---------------------------------")
print("checking duplicates...")
print("---------------------------------")

# CHECK DUPLICATE ENTRIES
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)

print(f"Duplicate entries before: {before}")
print(f"Duplicate entries after: {after}")
print(f"Duplicates removed: {before - after}")


Missing values per column:
label    0
text     0
dtype: int64

Shape after null drop: (5572, 2)
---------------------------------
checking duplicates...
---------------------------------
Duplicate entries before: 5572
Duplicate entries after: 5169
Duplicates removed: 403


In [9]:
# Encoding labels
# HAM = 0, SPAM = 1
df['label_num']= df['label'].map({'ham':0, 'spam':1})
print(df.head())

  label                                               text  label_num
0   ham  Go until jurong point, crazy.. Available only ...          0
1   ham                      Ok lar... Joking wif u oni...          0
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...          1
3   ham  U dun say so early hor... U c already then say...          0
4   ham  Nah I don't think he goes to usf, he lives aro...          0


    Full NLP preprocessing pipeline:
      1. Lowercase
      2. Remove URLs
      3. Remove digits
      4. Remove punctuation
      5. Strip extra whitespace
      6. Remove stopwords and single characters


In [10]:
# Text Cleaning

def clean_text(text):

    text=text.lower() # convert to lowercase
    text=re.sub(r"http\S+|www\S+", " ", text) # remove URLs
    text=re.sub(r"\d+", " ", text) # remove numbers
    text=text.translate(str.maketrans("", "", string.punctuation)) # remove punctuation
    text=re.sub(r"\s+", " ", text).strip() # remove extra whitespace   
    tokens=text.split() # tokenize
    tokens=[t for t in tokens if t not in STOP_WORDS and len(t) > 1] # remove stopwords and single characters
    return " ".join(tokens) 

# Apply to dataframe 
df['clean_text'] = df['text'].apply(clean_text)
print("Sample — original vs cleaned:")
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"  ORIGINAL : {row['text'][:80]}")
    print(f"  CLEANED  : {row['clean_text'][:80]}")
    print()


Sample — original vs cleaned:
  ORIGINAL : K, makes sense, btw carlos is being difficult so you guys are gonna smoke while 
  CLEANED  : makes sense btw carlos difficult guys gonna smoke go pick second batch get gas

  ORIGINAL : URGENT! Your mobile No *********** WON a £2,000 Bonus Caller Prize on 02/06/03! 
  CLEANED  : urgent mobile bonus caller prize nd attempt reach call asap box qp ppm

  ORIGINAL : If you still havent collected the dough pls let me know so i can go to the place
  CLEANED  : still havent collected dough pls let know go place sent get control number



In [11]:
# cleaned Summary
print(f"Final shape : {df.shape}")
print(f"\nLabel distribution after cleaning:")
print(df['label'].value_counts())
print(f"\nSpam ratio  : {df['label'].eq('spam').mean()*100:.1f}%")

Final shape : (5169, 4)

Label distribution after cleaning:
label
ham     4516
spam     653
Name: count, dtype: int64

Spam ratio  : 12.6%


Feature Engineering


In [13]:
# length of messages

df["char_count"]   = df["text"].apply(len)
df["word_count"]   = df["text"].apply(lambda x: len(x.split()))
df["clean_words"]  = df["clean_text"].apply(lambda x: len(x.split()))

df["avg_word_len"] = df["text"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
)

In [15]:
# Spam-Signal Analysis

df["url"]       = df["text"].str.contains(r"http|www|\.com", case=False).astype(int)
df["number"]    = df["text"].str.contains(r"\d", case=False).astype(int)
df["exclaim_count"] = df["text"].str.count(r"!")

df["uppercase_ratio"]   = df["text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)
)

In [23]:
# observation 
print("Feature columns added:")
print("---------------------------------------------------------------------------")
print(df[["label","char_count","word_count","clean_words",
          "avg_word_len","url","number","exclaim_count","uppercase_ratio"]].head())

Feature columns added:
---------------------------------------------------------------------------
  label  char_count  word_count  clean_words  avg_word_len  url  number  \
0   ham         111          20           14      4.600000    0       0   
1   ham          29           6            5      4.000000    0       0   
2  spam         155          28           19      4.571429    0       1   
3   ham          49          11            6      3.545455    0       0   
4   ham          61          13            8      3.769231    0       0   

   exclaim_count  uppercase_ratio  
0              0         0.027027  
1              0         0.068966  
2              0         0.064516  
3              0         0.040816  
4              0         0.032787  


In [24]:
# Average feature values by label

feat_cols = ["char_count",
            "word_count",
            "avg_word_len",
            "url", 
            "number", "exclaim_count",
            "uppercase_ratio"]

df.groupby("label")[feat_cols].mean().round(3)

,char_count,word_count,avg_word_len,url,number,exclaim_count,uppercase_ratio
label,,,,,,,
ham,70.906,14.24,4.146,0.003,0.158,0.178,0.057
spam,137.704,23.74,4.977,0.161,0.940,0.698,0.110
